# Full DRACO pipeline through the ScreamingFace SDK

This is the SDK-native port of the full `pipeline_walkthrough.ipynb` in
`screamingface-benchmarks/notebooks/general/`.
It preserves the published Candidate surface—**7 solo Models and 9 Fusions**—using only the public
SDK. The Engine owns DRACO's dataset, judge, grading, and aggregation. Each SDK Candidate owns its
answer and synthesis policy.

This notebook always selects canonical `draco`: all 100 Cases, every criterion, and five Judge
passes per criterion. It has no smoke or lite switch, so a completed Report is unambiguously a
full Engine-protocol result.

> **Fidelity status:** `full` describes multiplicity, not byte-for-byte reference conformance.
> The Engine uses the DRACO dataset source, official per-criterion Judge instructions, published
> scoring math, reference answer/synthesis system prompts, and the 7-solo/9-Fusion lineup. Do not
> present its score as a reproduced paper or OpenRouter-blog number yet:
>
> - canonical Engine DRACO uses the paper's five Judge passes, while the current source walkthrough
>   uses three; the retired paper Judge is replaced by Gemini 3.1 Pro and `reasoning=low` is not yet
>   forwarded on that OpenRouter route;
> - answer retrieval is route-declared provider-native search or a Tavily search/fetch loop, not the
>   reference harness's single OpenRouter server-tool configuration and exact tool budgets;
> - Fusion writers receive the same question and ordered labeled panel answers, but the universal
>   SDK framing omits the reference template's final redundant “Produce the unified prose answer
>   now.” sentence and labels members by their public Recipe names;
> - the reference harness can reuse prior solo answers in Fusion panels; this run currently makes
>   independent Candidate calls. That can change spend and generated answers, although the grading
>   protocol is unchanged.

> **Spend warning:** execution is disabled by default. Set `RUN_EVALUATION = True` only after
> reviewing the full experiment's estimated scope.

## Before running

Install the SDK with the runtime and notebook extras, then prepare and start the local Engine from a
terminal:

```bash
pip install "screamingface[runtime,notebook]"
screamingface prepare draco  # first run only: download pinned Benchmark assets
screamingface up             # start AI Gateway :9105 and Engine :9108
screamingface status
```

Use `screamingface down` when finished. Stack management stays outside the notebook so **Run All**
never starts or stops local services.

Export `TAVILY_API_KEY` before `screamingface up` if any provider route in your
local Engine needs web retrieval but does not offer provider-native web search. The Engine uses
Tavily for search and fetch on those routes and fails before model spend when that required
retrieval mechanism is unavailable.

In [ ]:
import screamingface as sf

## 1. Connect OpenRouter

In [ ]:
sf.connect()

## 2. Define the full solo lineup

These are the seven solo Candidates from the original full-pipelines notebook. Qwen is also
defined because it participates in the open-source Fusion. The reference configuration requires
at least 8,192 output tokens for answer and tool paths, so this notebook pins that Candidate policy
instead of inheriting the SDK's smaller general default.

The Gateway's live model contract currently marks `temperature` unsupported for Fable 5 and
GPT-5.5. Their requests therefore omit it; every model that supports it retains the reference
temperature of zero.

In [ ]:
DRACO_ANSWER_PROMPT = (
    "You are answering a research-quality prompt. Provide a thorough, "
    "well-reasoned answer in prose. Address every aspect the prompt raises. "
    "Use clear structure (headings, bullet lists where appropriate) and cite "
    "specific facts, methodologies, or sources where relevant.\n\n"
    "Do not refuse, abstain, or claim uncertainty unless the question is "
    "genuinely ambiguous — the goal is to demonstrate depth of understanding. "
    "Length: aim for the level of detail the question warrants; brevity that "
    "skips key points will be penalised by the rubric."
)

DRACO_SYNTHESIS_PROMPT = (
    "You are synthesising a single, comprehensive answer to a research-quality "
    "prompt by combining N independent answers from a panel of models. The "
    "downstream grader will score your output against a STRUCTURED RUBRIC of "
    "weighted criteria — your goal is to maximise rubric coverage.\n\n"
    "Procedure:\n"
    "1. Read every panel answer carefully.\n"
    "2. Identify which claims, facts, citations, or arguments each panel member "
    "contributes that the others miss.\n"
    "3. Produce ONE unified prose response that:\n"
    "   - Combines the strongest reasoning from every panel member\n"
    "   - Preserves specific named entities, dates, methodologies, and citations\n"
    "   - Resolves disagreements by favouring the more specific / better-cited claim\n"
    "   - Uses clear structure (headings, lists) where it aids the reader\n"
    "4. Do not introduce new facts that no panel member provided.\n"
    "5. Do not hedge or refuse — the panel collectively has enough material.\n\n"
    "Output: the unified prose answer, no preamble, no JSON wrapper."
)

In [ ]:
DRACO_PARAMS = {"max_tokens": 8192, "temperature": 0.0}
DRACO_PARAMS_NO_TEMPERATURE = {"max_tokens": 8192}

fable = sf.Model(
    "openrouter/anthropic/claude-fable-5",
    prompt=DRACO_ANSWER_PROMPT,
    params=DRACO_PARAMS_NO_TEMPERATURE,
)
opus = sf.Model(
    "openrouter/anthropic/claude-opus-4.8",
    prompt=DRACO_ANSWER_PROMPT,
    params=DRACO_PARAMS,
)
gpt = sf.Model(
    "openrouter/openai/gpt-5.5",
    prompt=DRACO_ANSWER_PROMPT,
    params=DRACO_PARAMS_NO_TEMPERATURE,
)
gemini_pro = sf.Model(
    "openrouter/google/gemini-3.1-pro-preview",
    prompt=DRACO_ANSWER_PROMPT,
    params=DRACO_PARAMS,
)
gemini_flash = sf.Model(
    "openrouter/google/gemini-3-flash-preview",
    prompt=DRACO_ANSWER_PROMPT,
    params=DRACO_PARAMS,
)
kimi = sf.Model(
    "openrouter/moonshotai/kimi-k2.6",
    prompt=DRACO_ANSWER_PROMPT,
    params=DRACO_PARAMS,
)
deepseek = sf.Model(
    "openrouter/deepseek/deepseek-v4-pro",
    prompt=DRACO_ANSWER_PROMPT,
    params=DRACO_PARAMS,
)
qwen = sf.Model(
    "openrouter/qwen/qwen3.6-plus",
    prompt=DRACO_ANSWER_PROMPT,
    params=DRACO_PARAMS,
)

## 3. Define the nine Fusion Candidates

Each Fusion names the synthesizer from the reproduced configuration explicitly. Equivalent Models
deduplicate inside one Candidate graph. The self-Fusion uses explicit sample identities and
temperature so its two Opus calls remain independent. DRACO gives guarded retrieval to
answer-producing members; whole-Fusion synthesis and the Benchmark-owned Judge remain
retrieval-free.

The reference harness can reuse solo answers across overlapping Fusion panels. Until the Engine's
cross-Candidate cache lands, this SDK run evaluates each Candidate independently and may repeat
those member calls. The grading protocol stays fixed, but fresh provider calls can change both the
generated answer and spend; cross-harness scores therefore are not assumed identical.

In [ ]:
fable_plus_gpt = sf.Fusion(
    [fable, gpt],
    name="fable_plus_gpt",
    synthesizer=sf.Model(
        "openrouter/anthropic/claude-opus-4.8",
        prompt=DRACO_SYNTHESIS_PROMPT,
        params=DRACO_PARAMS,
    ),
)
frontier_trio = sf.Fusion(
    [opus, gpt, gemini_pro],
    name="frontier_trio",
    synthesizer=sf.Model(
        "openrouter/anthropic/claude-opus-4.8",
        prompt=DRACO_SYNTHESIS_PROMPT,
        params=DRACO_PARAMS,
    ),
)
opus_plus_gpt = sf.Fusion(
    [opus, gpt],
    name="opus_plus_gpt",
    synthesizer=sf.Model(
        "openrouter/anthropic/claude-opus-4.8",
        prompt=DRACO_SYNTHESIS_PROMPT,
        params=DRACO_PARAMS,
    ),
)
opus_self_fusion = sf.Fusion(
    [
        sf.Model(
            "openrouter/anthropic/claude-opus-4.8",
            name="opus-sample-1",
            prompt=DRACO_ANSWER_PROMPT,
            params={"max_tokens": 8192, "temperature": 0.7},
        ),
        sf.Model(
            "openrouter/anthropic/claude-opus-4.8",
            name="opus-sample-2",
            prompt=DRACO_ANSWER_PROMPT,
            params={"max_tokens": 8192, "temperature": 0.7},
        ),
    ],
    name="opus_self_fusion",
    synthesizer=sf.Model(
        "openrouter/anthropic/claude-opus-4.8",
        prompt=DRACO_SYNTHESIS_PROMPT,
        params=DRACO_PARAMS,
    ),
)
budget_trio = sf.Fusion(
    [gemini_flash, kimi, deepseek],
    name="budget_trio",
    synthesizer=sf.Model(
        "openrouter/anthropic/claude-opus-4.8",
        prompt=DRACO_SYNTHESIS_PROMPT,
        params=DRACO_PARAMS,
    ),
)
beat_runner_up = sf.Fusion(
    [opus, gpt, deepseek],
    name="beat_runner_up",
    synthesizer=sf.Model(
        "openrouter/anthropic/claude-opus-4.8",
        prompt=DRACO_SYNTHESIS_PROMPT,
        params=DRACO_PARAMS,
    ),
)
pareto_cross = sf.Fusion(
    [deepseek, kimi, gpt],
    name="pareto_cross",
    synthesizer=sf.Model(
        "openrouter/deepseek/deepseek-v4-pro",
        prompt=DRACO_SYNTHESIS_PROMPT,
        params=DRACO_PARAMS,
    ),
)
pareto_lean = sf.Fusion(
    [deepseek, kimi],
    name="pareto_lean",
    synthesizer=sf.Model(
        "openrouter/deepseek/deepseek-v4-pro",
        prompt=DRACO_SYNTHESIS_PROMPT,
        params=DRACO_PARAMS,
    ),
)
best_open_source = sf.Fusion(
    [deepseek, kimi, qwen],
    name="best_open_source",
    synthesizer=sf.Model(
        "openrouter/deepseek/deepseek-v4-pro",
        prompt=DRACO_SYNTHESIS_PROMPT,
        params=DRACO_PARAMS,
    ),
)

## 4. Arm the canonical run explicitly

In [ ]:
RUN_EVALUATION = False

## 5. Evaluate every Candidate

One Evaluation runs the complete Candidate lineup concurrently. Leaving
`RUN_EVALUATION = False` makes **Run All** safe and performs no model calls.

In [ ]:
candidates = [
    fable,
    opus,
    gpt,
    gemini_pro,
    gemini_flash,
    kimi,
    deepseek,
    fable_plus_gpt,
    frontier_trio,
    opus_plus_gpt,
    opus_self_fusion,
    budget_trio,
    beat_runner_up,
    pareto_cross,
    pareto_lean,
    best_open_source,
]

report = sf.evaluate(candidates, benchmark="draco") if RUN_EVALUATION else None
report_output = (
    report.to_json()
    if report is not None
    else "Evaluation disabled — set RUN_EVALUATION = True to spend."
)
report_output

## 6. Inspect the Report

The Report presents a typed leaderboard plus the exact Case artifacts, failures, operation graphs,
timing, usage, and portable JSON needed to audit the full experiment.

In [ ]:
if report is not None:
    leaderboard = sorted(
        (
            {
                "name": result.name,
                "kind": result.kind,
                "score": result.score,
                "duration_ms": result.duration_ms,
                "failures": len(result.failures),
                "input_tokens": result.usage.input_tokens,
                "output_tokens": result.usage.output_tokens,
                "cost_usd": result.usage.cost_usd,
            }
            for result in report.candidates
        ),
        key=lambda row: (row["score"] is not None, row["score"] or 0.0),
        reverse=True,
    )
else:
    leaderboard = []
leaderboard

In [ ]:
if report is not None:
    selected = report.candidates[0]
    selected_case = selected.cases[0]
    audit_sample = {
        "candidate": selected.name,
        "compiled_url4": selected.url4,
        "models": selected.models,
        "operations": selected.operations,
        "members": selected.members,
        "case_id": selected_case.case_id,
        "finish_reason": selected_case.finish_reason,
        "grade": selected_case.grade,
        "checks": () if selected_case.grade is None else selected_case.grade.checks,
        "case_failures": selected_case.failures,
    }
else:
    audit_sample = None
audit_sample

In [ ]:
{
    "ok": report.ok,
    "benchmark": report.benchmark,
    "case_count": report.case_count,
    "duration_ms": report.duration_ms,
    "failures": report.failures,
    "usage": report.usage,
} if report is not None else None

In [ ]:
report.to_json() if report is not None else None